# Mission Parameters Extraction

**Notebook 2c** - Data Cleaning Phase: Mission Parameters

## Purpose
Extract and clean mission-related parameters from raw Space Devs launch data, focusing on mission objectives, orbit details, and launch service provider information.

## Authors
- **Phillip Roman** - Mission parameter extraction and cleaning

## Workflow Position
1. Data Collection → Raw launch data collected via API
2. Data Cleaning → Extract parameters from raw data:
   - 2a. Extract rocket parameters
   - 2b. Extract launch parameters
   - 2c. **This Notebook** → Extract mission parameters (10 attributes)
3. Data Merging → Combine all cleaned datasets

## Key Features
- **Flexible Data Loading:** Automatically downloads data from GitHub if the local file isn't found (ensures the code works in Google Colab).
- **Automated Directory Creation:** Checks if the output folder exists and creates it if necessary to prevent file saving errors.
- **Navigating Nested Data:** Drills down into complex JSON dictionaries to extract specific details like orbit names and provider types.
- **Handling Missing Keys:** Safely manages records that are missing specific sections (like `program` or `orbit`) so the code doesn't crash on incomplete data.

## Key Parameters Extracted
- Mission identification and description
- Orbit specifications
- Program affiliations
- Launch service provider details

## Output
- `clean_mission_data.tsv` - Mission parameters ready for merging

## Import Required Libraries

Load necessary Python packages for data extraction and file operations.

In [1]:
from pprint import pprint
import pandas as pd
import zipfile
import requests
import json
import csv
import io
import os

## Load Raw Data

This cell implements a **hybrid loading strategy** to ensure reproducibility across different environments (Local vs. Cloud).

**Logic Flow:**
1.  **Attempt Local Load:** Checks the standard relative path (`../data/raw data/...`). If the file exists (typical for local development), it loads instantly.
2.  **Fallback to GitHub:** If the local file is missing (typical for Google Colab or fresh clones), it automatically streams the full dataset directly from the project's GitHub repository.

**Note:** This ensures the notebook runs immediately upon opening without requiring manual file uploads or drive mounting.

In [2]:
# primary - local path
zip_path = os.path.join("..", "data", "raw data", "raw_baseline_launches_Group7.json.zip")
launch_data_filename = 'raw_baseline_launches_Group7.json'

# backup - for cloud/colab
github_url = "https://github.com/Rybus07/space-legends-data/raw/main/data/raw%20data/raw_baseline_launches_Group7.json.zip"

print(f"Attempting to load data...")

try:
    print(f"Checking local path: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(launch_data_filename) as f:
            raw_launch_data = json.load(f)
    print("Success! Loaded from local file system.")

except FileNotFoundError:
    # stream form GitHub
    print("Local file not found (running in Cloud/Colab?).")
    print(f"Attempting download from GitHub: {github_url}")

    response = requests.get(github_url)
    if response.status_code == 200:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            with z.open(launch_data_filename) as f:
                raw_launch_data = json.load(f)
        print("Success! Loaded data directly from GitHub repository.")
    else:
        print(f"CRITICAL ERROR: Could not load data locally or from GitHub. Status: {response.status_code}")


Attempting to load data...
Checking local path: ../data/raw data/raw_baseline_launches_Group7.json.zip
Success! Loaded from local file system.


## Define Data Source Scope

This cell isolates the list of launch records from the loaded JSON object.

**Configuration Options:**
- **Default (Production):** `USE_SAMPLE = False` processes every record found in the loaded file (7,000+ launches).
- **Debugging (Optional):** Set `USE_SAMPLE = True` to process only the last 50 records. This is useful for rapidly testing changes to the extraction loop logic without waiting for the full dataset.

In [3]:
all_launches = raw_launch_data.get('launches', [])
print(f"Loaded {len(all_launches)} total launches.")

USE_SAMPLE = False

if USE_SAMPLE:
    data_source = all_launches[-50:] # Sample last 50
else:
    data_source = all_launches # Full dataset

print(f"Processing {len(data_source)} launches...")

Loaded 7336 total launches.
Processing 7336 launches...


## Inspect Raw Data Structure

This cell prints the last record from the `data_source` list to visualize the raw JSON structure.

**Purpose:**
- **Key Discovery:** Allows for inspection of available dictionary keys (e.g., `mission`, `orbit`, `program`) to guide the extraction logic.
- **Data Validation:** Confirms that the loaded data contains the expected nested objects before processing begins.

In [4]:
pprint(data_source[-1])

{'agency_launch_attempt_count': 596,
 'agency_launch_attempt_count_year': 147,
 'failreason': '',
 'flightclub_url': 'https://flightclub.io/result?llId=6602c88f-cbff-4495-b417-a184ddb0a426',
 'hashtag': None,
 'id': '6602c88f-cbff-4495-b417-a184ddb0a426',
 'image': {'credit': 'SpaceX',
           'id': 1296,
           'image_url': 'https://thespacedevs-prod.nyc3.digitaloceanspaces.com/media/images/falcon2520925_image_20221009234147.png',
           'license': {'id': 5,
                       'link': 'https://creativecommons.org/licenses/by-nc/2.0/',
                       'name': 'CC BY-NC 2.0',
                       'priority': 1},
           'name': 'Starlink night fairing',
           'single_use': False,
           'thumbnail_url': 'https://thespacedevs-prod.nyc3.digitaloceanspaces.com/media/images/255bauto255d__image_thumbnail_20240305192320.png',
           'variants': []},
 'info_urls': [{'description': 'SpaceX designs, manufactures and launches '
                             

## Prototype Extraction Logic

This cell validates the extraction strategy on a single launch record before applying it to the full dataset.

**Key Objectives:**
1.  **Path Verification:** Confirms that the target JSON paths (e.g., `mission` $\rightarrow$ `orbit` $\rightarrow$ `name`) correctly map to the desired data.
2.  **Safe Unnesting:** Tests a defensive coding technique using chained `.get()` methods (e.g., `.get('mission', {}).get('name')`). This ensures the code returns `None` rather than crashing if a nested dictionary is missing.

*Reference logic adapted from [Stack Overflow: Safe method to get value of nested dictionary](https://stackoverflow.com/questions/25833613/safe-method-to-get-value-of-nested-dictionary).*

In [5]:
test_launch = data_source[0]

print("Testing 'Mission' fields:")
print(f"Launch ID: {test_launch.get('id')}")
print(f"Mission Name: {test_launch.get('mission', {}).get('name')}")
print(f"Mission Type: {test_launch.get('mission', {}).get('type')}")
print(f"Orbit Name: {test_launch.get('mission', {}).get('orbit', {}).get('name')}")

print("\nTesting 'Crew/Program' field:")
print(f"Program: {test_launch.get('program')}") # shows it's a list

print("\nTesting 'Agency/LSP' fields:")
print(f"LSP Name: {test_launch.get('launch_service_provider', {}).get('name')}")
print(f"LSP Type: {test_launch.get('launch_service_provider', {}).get('type', {}).get('name')}")

Testing 'Mission' fields:
Launch ID: e3df2ecd-c239-472f-95e4-2b89b4f75800
Mission Name: Sputnik 1
Mission Type: Test Flight
Orbit Name: Low Earth Orbit

Testing 'Crew/Program' field:
Program: []

Testing 'Agency/LSP' fields:
LSP Name: Soviet Space Program
LSP Type: Government


## Extract Mission and Agency Parameters

This cell executes the core processing loop to extract 10 key attributes related to the mission objectives and launch providers.

**Key Extraction Logic:**
1.  **Extracting Nested Data:** Drills down into dictionary structures (e.g., `launch` $\rightarrow$ `mission` $\rightarrow$ `orbit`) to retrieve specific details like orbit names.
2.  **Handling Missing Values:** Uses `if/else` logic to check if data exists before trying to access it. If a section (like `launch_service_provider`) is missing, we explicitly assign `None` to ensure every row has the same number of columns.
3.  **Processing Lists:** Since the `program` field is a list, we extract just the first item (index 0) to get the primary program name associated with the launch.

### Parameters Extracted:
1.  `launch_id` - Primary key for merging
2.  `mission_id` - Mission identifier
3.  `mission_name` - Mission designation
4.  `mission_type` - Category (Test Flight, Communications, etc.)
5.  `mission_description` - Detailed mission objective
6.  `orbit_name` - Target orbit (Low Earth Orbit, GTO, etc.)
7.  `orbit_abbrev` - Orbit abbreviation (LEO, GTO, etc.)
8.  `program_name` - Associated program (ISS, Artemis, etc.)
9.  `lsp_name` - Launch Service Provider name
10. `lsp_type` - Provider type (Government, Commercial, etc.)

In [6]:
mission_table = []
mission_header = [
    "launch_id",
    "mission_id",
    "mission_name",
    "mission_type",
    "mission_description",
    "orbit_name",
    "orbit_abbrev",
    "program_name",
    "lsp_name",
    "lsp_type"
]

for launch in data_source:

    launch_id = launch.get('id')

    # mission and orbit data
    mission_data = launch.get('mission')
    if mission_data:
        mission_id = mission_data.get('id')
        mission_name = mission_data.get('name')
        mission_type = mission_data.get('type')
        mission_desc = mission_data.get('description')

        # orbit data nested inside mission
        orbit_data = mission_data.get('orbit')
        if orbit_data:
            orbit_name = orbit_data.get('name')
            orbit_abbrev = orbit_data.get('abbrev')
        else:
            orbit_name = None
            orbit_abbrev = None
    else:
        # handle missing Mission data
        mission_id = None
        mission_name = None
        mission_type = None
        mission_desc = None
        orbit_name = None
        orbit_abbrev = None

    program_list = launch.get('program', [])
    if program_list:
        program_name = program_list[0].get('name') # first program only
    else:
        program_name = None

    # launch service provider "agency"
    lsp_data = launch.get('launch_service_provider')
    if lsp_data:
        lsp_name = lsp_data.get('name')

        lsp_type_data = lsp_data.get('type')
        if lsp_type_data:
            lsp_type = lsp_type_data.get('name')
        else:
            lsp_type = None
    else:
        lsp_name = None
        lsp_type = None

    row = [
        launch_id,
        mission_id,
        mission_name,
        mission_type,
        mission_desc,
        orbit_name,
        orbit_abbrev,
        program_name,
        lsp_name,
        lsp_type
    ]
    mission_table.append(row)

print(f"Finished mission parameter extraction. Created table with {len(mission_table)} rows.")

Finished mission parameter extraction. Created table with 7336 rows.


#### Create DataFrame and Analyze Mission Distributions

This cell converts the extracted list into a pandas DataFrame and performs an initial categorical analysis.

**Key Insights:**
1.  **Mission Categorization:** Analyzes the `mission_type` distribution to understand the primary purposes of launches (e.g., Communications, Earth Science).
2.  **Sector Breakdown:** Examines `lsp_type` to compare the volume of Government versus Commercial launches.
3.  **Data Validation:** Displays a sample of rows and the DataFrame structure to verify data types and identify missing values (nulls) in the new dataset.

In [7]:
df_mission = pd.DataFrame(mission_table, columns=mission_header)
print("Successfully created Mission DataFrame:")

print("\nMission Type Breakdown:")
print(df_mission['mission_type'].value_counts())

print("\nLSP Type Breakdown:")
print(df_mission['lsp_type'].value_counts())


print("\nPrinting first 3 rows:")
print(df_mission.head(3))


print("\nDataFrame Info (Checking for nulls)")
df_mission.info()

Successfully created Mission DataFrame:

Mission Type Breakdown:
mission_type
Government/Top Secret          2253
Communications                 1570
Earth Science                   936
Navigation                      407
Test Flight                     342
Human Exploration               314
Test Target                     205
Astrophysics                    167
Resupply                        124
Robotic Exploration              88
Lunar Exploration                88
Dedicated Rideshare              67
Planetary Science                59
Technology                       43
Heliophysics                     37
                                 26
Tourism                          25
Materials Science                24
Suborbital                       22
Biology                          18
Unknown                           3
Space Situational Awareness       1
Name: count, dtype: int64

LSP Type Breakdown:
lsp_type
Government       5307
Commercial       1945
Private            72
Multinat

## Preview Final Dataset

This cell displays the first 10 rows of the `df_mission` DataFrame to validate the extraction results.

**Key Observations:**
- **Schema Validation:** Checks that all 10 columns (`launch_id` through `lsp_type`) are present and correctly ordered.
- **Data Integrity:** Allows for a visual spot-check of complex text fields (like `mission_description`) to ensure no character encoding errors or truncation occurred during the extraction process.

In [8]:
df_mission.head(10)

,launch_id,mission_id,mission_name,mission_type,mission_description,orbit_name,orbit_abbrev,program_name,lsp_name,lsp_type
0,e3df2ecd-c239-472f-95e4-2b89b4f75800,1430.0,Sputnik 1,Test Flight,First artificial satellite consisting of a 58 ...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
1,f8c9f344-a6df-4f30-873a-90fe3a7840b3,1431.0,Sputnik 2,Test Flight,Second artificial satellite and first to carry...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
2,535c1a09-97c8-4f96-bb64-6336d4bcb1fb,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
3,1b9e28d0-c531-44b0-9b37-244e62a6d3f4,1433.0,Explorer 1,Test Flight,First successfully launched American satellite...,Low Earth Orbit,LEO,None,Army Ballistic Missile Agency,Government
4,48bc7deb-b2e1-46c2-ab63-0ce00fbd192b,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
5,896e8af6-d256-4a5b-ab15-2f25c84e90e3,1434.0,Explorer 2,Test Flight,Small satellite similar to Explorer 1. It fail...,Low Earth Orbit,LEO,None,Army Ballistic Missile Agency,Government
6,74d39bb8-34a6-4a8b-8554-d2d3ec22aee6,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government
7,b4e501ff-083c-47d6-9ff0-63ec1bf035c3,1435.0,Explorer 3,Earth Science,Small satellite launched into an eccentric orb...,Elliptical Orbit,Elliptical,None,Army Ballistic Missile Agency,Government
8,59d2de37-4c22-495f-8718-4b22f5f34ab7,1436.0,D-1 1,Earth Science,First complex scientific satellite with 12 exp...,Low Earth Orbit,LEO,None,Soviet Space Program,Government
9,de282e74-e03b-411e-9633-2d1497629893,1432.0,Vanguard,Test Flight,Small satellite used to test the Vanguard thre...,Low Earth Orbit,LEO,None,US Navy,Government


## Save Cleaned Data

This cell saves the processed dataframe to a TSV file.

**Features:**
- **Automatic Directory Creation:** Checks if the target folder (`../data/cleaned data/`) exists and creates it if missing.
- **TSV Format:** Saves as Tab-Separated Values to safely handle text fields containing commas (common in Mission Descriptions).

In [9]:
output_filename = "clean_mission_data.tsv"
save_path = os.path.join("..", "data", "cleaned data", output_filename)

print(f"Attempting to save data to: {save_path}")

# checks if directory exists
folder_path = os.path.dirname(save_path)
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Created missing directory structure: {folder_path}")

# saves file
try:
    df_mission.to_csv(save_path, sep='\t', index=False)
    print(f"Success! Data saved to: {save_path}")
    print(f"File size: {os.path.getsize(save_path) / 1024:.2f} KB")
except Exception as e:
    print(f"Error saving file: {e}")

Attempting to save data to: ../data/cleaned data/clean_mission_data.tsv
Success! Data saved to: ../data/cleaned data/clean_mission_data.tsv
File size: 2074.40 KB
